# First Byte — Team Integration

Purpose:
Combine monthly directional-route revenue and operating costs.
Add gate-rent allocation when available.

Current approach:
Use revenue lower and upper bounds to calculate operating result bounds.
Confirmed revenue is a partial amount, not complete revenue.

Limitations:
Revenue uses transaction processing dates.
Costs use flight cost dates.
Results are period-level financial comparisons, not matched flight profits.
Monthly bounds must not be summed as exact annual revenue.

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

pd.options.display.float_format = "{:,.2f}".format

print("Analysis tools loaded.")

In [ ]:
uploaded = files.upload()

print("Uploaded files:")
for filename in uploaded:
    print(filename)

In [ ]:
uploaded = files.upload()

print("Uploaded files:")
for filename in uploaded:
    print(filename)

In [ ]:
# Read the files already uploaded to the runtime.
cost_input = pd.read_csv(
    "route_monthly_costs (1).csv",
    encoding="utf-8-sig"
)

revenue_input = pd.read_csv(
    "route_month_revenue.csv",
    encoding="utf-8-sig"
)

print("Cost table shape:", cost_input.shape)
print("Revenue table shape:", revenue_input.shape)

print("\nCost table preview:")
display(cost_input.head())

print("\nRevenue table preview:")
display(revenue_input.head())

In [ ]:
# Preserve the original input tables.
cost_work = cost_input.copy()
revenue_work = revenue_input.copy()

# Normalize month labels using the known input formats.
for table, date_format in [
    (cost_work, "%Y-%m"),
    (revenue_work, "%Y-%m-%d")
]:
    month_text = table["Month"].astype("string").str.strip()

    parsed_month = pd.to_datetime(
        month_text,
        format=date_format,
        errors="coerce"
    )

    assert parsed_month.notna().all(), (
        "Missing or invalid month labels found."
    )

    table["Month"] = (
        parsed_month.dt.to_period("M").astype(str)
    )

print("Cost month labels:")
display(cost_work[["Month"]].head())

print("Revenue month labels:")
display(revenue_work[["Month"]].head())

In [ ]:
# Normalize directional route labels.
for table in [cost_work, revenue_work]:
    table["Route_ID_Original"] = table["Route_ID"]

    compact_route = (
        table["Route_ID"]
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"[^A-Z]", "", regex=True)
    )

    valid_route = compact_route.str.fullmatch(
        r"[A-Z]{6}",
        na=False
    )

    if not valid_route.all():
        display(
            table.loc[
                ~valid_route,
                ["Month", "Route_ID_Original"]
            ].head(20)
        )
        raise ValueError("Review invalid route labels.")

    table["Route_ID"] = (
        compact_route.str[:3]
        + "-"
        + compact_route.str[3:]
    )

print("Standardized cost keys:")
display(cost_work[["Month", "Route_ID"]].head())

print("Standardized revenue keys:")
display(revenue_work[["Month", "Route_ID"]].head())

In [ ]:
# Validate unique merge keys.
merge_keys = ["Month", "Route_ID"]

for table_name, table in [
    ("Cost", cost_work),
    ("Revenue", revenue_work)
]:
    duplicate_keys = table.duplicated(
        merge_keys,
        keep=False
    )

    print(
        f"{table_name} rows with duplicate keys:",
        int(duplicate_keys.sum())
    )

    if duplicate_keys.any():
        display(
            table.loc[duplicate_keys]
            .sort_values(merge_keys)
            .head(20)
        )
        raise ValueError(
            f"Duplicate month-route keys found in {table_name}."
        )

print("Merge key checks passed.")

In [ ]:
# Merge all observed month-route combinations.
integration = cost_work.merge(
    revenue_work,
    on=merge_keys,
    how="outer",
    validate="one_to_one",
    indicator="MatchStatus",
    suffixes=("_Cost", "_Revenue")
)

integration = (
    integration
    .sort_values(merge_keys)
    .reset_index(drop=True)
)

print("Match status counts:")
display(integration["MatchStatus"].value_counts())

unmatched_rows = integration.loc[
    integration["MatchStatus"] != "both"
]

print("\nUnmatched row count:", len(unmatched_rows))

display(
    unmatched_rows[
        [
            "Month",
            "Route_ID",
            "MatchStatus",
            "OperatingCost",
            "net_revenue_lower_cad",
            "net_revenue_upper_cad"
        ]
    ].head(20)
)

In [ ]:
# Verify that merging preserved the source totals.
checks = [
    (
        "OperatingCost",
        cost_work["OperatingCost"].sum(),
        integration["OperatingCost"].sum()
    ),
    (
        "RevenueLower",
        revenue_work["net_revenue_lower_cad"].sum(),
        integration["net_revenue_lower_cad"].sum()
    ),
    (
        "RevenueUpper",
        revenue_work["net_revenue_upper_cad"].sum(),
        integration["net_revenue_upper_cad"].sum()
    )
]

for label, before, after in checks:
    assert np.isclose(
        before,
        after,
        rtol=0,
        atol=0.01
    ), f"{label} changed during merging."

    print(f"{label} preserved: True")

print("Merge reconciliation passed.")

In [ ]:
# Inspect every revenue-only month-route combination.
revenue_only = integration.loc[
    integration["MatchStatus"] == "right_only"
].copy()

review_columns = [
    "Month",
    "Route_ID",
    "net_revenue_confirmed_cad",
    "net_revenue_lower_cad",
    "net_revenue_upper_cad",
    "confirmed_transaction_count",
    "possible_ambiguous_transaction_count"
]

display(revenue_only[review_columns])

print("Revenue-only rows:", len(revenue_only))

In [ ]:
# Preserve the merged table before handling absent cost records.
profit_bounds = integration.copy()

profit_bounds["NoObservedFlightCost"] = (
    profit_bounds["MatchStatus"] == "right_only"
)

# Matched rows must already contain valid costs and flight counts.
matched_rows = profit_bounds["MatchStatus"] == "both"

assert profit_bounds.loc[
    matched_rows,
    ["OperatingCost", "FlightCount"]
].notna().all().all(), "Matched rows contain missing cost data."

assert not (
    profit_bounds["MatchStatus"] == "left_only"
).any(), "Review cost-only rows before continuing."

# Fill costs only where the cost table has no matching record.
profit_bounds.loc[
    profit_bounds["NoObservedFlightCost"],
    ["OperatingCost", "FlightCount"]
] = 0

profit_bounds["FlightCount"] = (
    profit_bounds["FlightCount"].astype(int)
)

assert np.isclose(
    profit_bounds["OperatingCost"].sum(),
    cost_work["OperatingCost"].sum(),
    rtol=0,
    atol=0.01
), "Operating costs changed during missing-record handling."

print("Revenue-only rows retained:",
      int(profit_bounds["NoObservedFlightCost"].sum()))

print("Operating cost total preserved.")

In [ ]:
# Validate the revenue bounds.
revenue_bound_columns = [
    "net_revenue_lower_cad",
    "net_revenue_upper_cad"
]

for column in revenue_bound_columns:
    profit_bounds[column] = pd.to_numeric(
        profit_bounds[column],
        errors="coerce"
    )

assert profit_bounds[revenue_bound_columns].notna().all().all(), (
    "Missing or invalid revenue bounds found."
)

assert np.isfinite(
    profit_bounds[revenue_bound_columns].to_numpy()
).all(), "Non-finite revenue bounds found."

assert (
    profit_bounds["net_revenue_lower_cad"]
    <= profit_bounds["net_revenue_upper_cad"] + 0.01
).all(), "Revenue lower bounds exceed upper bounds."

# Calculate period-level operating result bounds before gate rent.
profit_bounds["OperatingResultLower"] = (
    profit_bounds["net_revenue_lower_cad"]
    - profit_bounds["OperatingCost"]
)

profit_bounds["OperatingResultUpper"] = (
    profit_bounds["net_revenue_upper_cad"]
    - profit_bounds["OperatingCost"]
)

print("Operating result bounds calculated.")

In [ ]:
# Classify matched rows using a one-cent tolerance.
tolerance = 0.01

profit_bounds["ResultStatus"] = np.select(
    [
        profit_bounds["NoObservedFlightCost"],
        profit_bounds["OperatingResultUpper"] < -tolerance,
        profit_bounds["OperatingResultLower"] > tolerance
    ],
    [
        "No observed flight cost",
        "Loss under all date allocations",
        "Profit under all date allocations"
    ],
    default="Break-even or date-sensitive"
)

print("Result status counts:")
display(profit_bounds["ResultStatus"].value_counts())

# Review matched rows with the most negative upper bounds.
robust_loss_rows = (
    profit_bounds.loc[
        profit_bounds["ResultStatus"]
        == "Loss under all date allocations"
    ]
    .sort_values("OperatingResultUpper")
)

display(
    robust_loss_rows[
        [
            "Month",
            "Route_ID",
            "FlightCount",
            "net_revenue_lower_cad",
            "net_revenue_upper_cad",
            "OperatingCost",
            "OperatingResultLower",
            "OperatingResultUpper"
        ]
    ].head(10).round(2)
)

In [ ]:
from google.colab import files

audit_upload = files.upload()

print("Uploaded audit files:")
for filename in audit_upload:
    print(filename)

In [ ]:
import io
import pandas as pd
import numpy as np

# Read the exact files from the latest upload.
transactions_audit = pd.read_csv(
    io.BytesIO(
        audit_upload["transactions_deduplicated_audit.csv"]
    ),
    encoding="utf-8-sig",
    dtype={"Transaction_ID": "string", "Route_ID": "string"}
)

date_candidates = pd.read_csv(
    io.BytesIO(
        audit_upload["transaction_date_candidates.csv"]
    ),
    encoding="utf-8-sig",
    dtype={"Transaction_ID": "string"}
)

# Check required columns.
required_transaction_columns = {
    "Transaction_ID",
    "Route_ID",
    "Amount",
    "Date_Status",
    "source_record_count"
}

required_candidate_columns = {
    "Transaction_ID",
    "candidate_date",
    "candidate_month",
    "candidate_fiscal_year"
}

assert required_transaction_columns.issubset(
    transactions_audit.columns
), "Missing transaction columns."

assert required_candidate_columns.issubset(
    date_candidates.columns
), "Missing candidate columns."

# Validate transaction IDs and amounts.
assert transactions_audit["Transaction_ID"].notna().all()
assert transactions_audit["Transaction_ID"].is_unique

transactions_audit["Amount"] = pd.to_numeric(
    transactions_audit["Amount"],
    errors="raise"
)

assert np.isfinite(transactions_audit["Amount"]).all()

# Validate date candidate coverage.
assert date_candidates["Transaction_ID"].notna().all()

assert not date_candidates.duplicated(
    ["Transaction_ID", "candidate_date"]
).any(), "Duplicate date candidates."

assert set(transactions_audit["Transaction_ID"]) == set(
    date_candidates["Transaction_ID"]
), "Transaction IDs do not match."

assert len(transactions_audit) == 558129
assert len(date_candidates) == 564189

assert np.isclose(
    transactions_audit["Amount"].sum(),
    370629024.85,
    rtol=0,
    atol=0.01
), "Net revenue total does not match."

print("Audit checks passed.")
print(f"Unique transactions: {len(transactions_audit):,}")
print(f"Date candidate rows: {len(date_candidates):,}")
print(
    "Total net transaction revenue: "
    f"CAD {transactions_audit['Amount'].sum():,.2f}"
)

print("\nTransaction date status:")
display(
    transactions_audit["Date_Status"]
    .value_counts()
    .rename_axis("DateStatus")
    .to_frame("TransactionCount")
)

print("\nTransaction sample:")
display(
    transactions_audit[
        ["Transaction_ID", "Route_ID", "Amount", "Date_Status"]
    ].head()
)

print("\nDate candidate sample:")
display(
    date_candidates[
        [
            "Transaction_ID",
            "candidate_date",
            "candidate_month",
            "candidate_fiscal_year"
        ]
    ].head()
)

In [ ]:
# Keep each transaction only once per possible fiscal year.
transaction_fy = (
    date_candidates[
        ["Transaction_ID", "candidate_fiscal_year"]
    ]
    .drop_duplicates()
    .rename(columns={"candidate_fiscal_year": "FY"})
    .copy()
)

assert transaction_fy["FY"].notna().all()
assert set(transaction_fy["FY"]) == {
    "FY2024-25",
    "FY2025-26"
}

# Count the number of possible fiscal years per transaction.
transaction_fy["PossibleFYCount"] = (
    transaction_fy.groupby("Transaction_ID")["FY"]
    .transform("size")
)

assert transaction_fy["PossibleFYCount"].isin([1, 2]).all()

# Attach each transaction's route and amount.
transaction_fy = transaction_fy.merge(
    transactions_audit[
        ["Transaction_ID", "Route_ID", "Amount"]
    ],
    on="Transaction_ID",
    how="left",
    validate="many_to_one"
)

assert transaction_fy["Amount"].notna().all()

# Standardize route IDs to match the operating cost table.
route_codes = (
    transaction_fy["Route_ID"]
    .str.upper()
    .str.replace(r"[^A-Z]", "", regex=True)
)

assert route_codes.str.fullmatch(r"[A-Z]{6}").all()

transaction_fy["Route_ID"] = (
    route_codes.str[:3] + "-" + route_codes.str[3:]
)

# A transaction is certain for a fiscal year if it has only one possible FY.
fy_is_certain = transaction_fy["PossibleFYCount"].eq(1)

transaction_fy["NetRevenueConfirmed"] = np.where(
    fy_is_certain,
    transaction_fy["Amount"],
    0.0
)

# For uncertain FY assignments:
# positive amounts affect the upper bound;
# negative amounts affect the lower bound.
transaction_fy["NetRevenueLower"] = np.where(
    fy_is_certain,
    transaction_fy["Amount"],
    np.minimum(transaction_fy["Amount"], 0.0)
)

transaction_fy["NetRevenueUpper"] = np.where(
    fy_is_certain,
    transaction_fy["Amount"],
    np.maximum(transaction_fy["Amount"], 0.0)
)

# Summarize revenue bounds by fiscal year and directional route.
route_fiscal_revenue_bounds = (
    transaction_fy.groupby(
        ["FY", "Route_ID"],
        as_index=False
    )[
        [
            "NetRevenueConfirmed",
            "NetRevenueLower",
            "NetRevenueUpper"
        ]
    ]
    .sum()
    .sort_values(["FY", "Route_ID"])
    .reset_index(drop=True)
)

assert (
    route_fiscal_revenue_bounds["NetRevenueLower"]
    <= route_fiscal_revenue_bounds["NetRevenueUpper"] + 0.01
).all()

# Reconcile the route totals to the previously verified FY bounds.
fiscal_revenue_check = (
    route_fiscal_revenue_bounds.groupby("FY")[
        ["NetRevenueLower", "NetRevenueUpper"]
    ]
    .sum()
    .sort_index()
)

expected_fiscal_bounds = pd.DataFrame(
    {
        "NetRevenueLower": [179939246.53, 189253309.95],
        "NetRevenueUpper": [181375714.90, 190689778.32]
    },
    index=pd.Index(
        ["FY2024-25", "FY2025-26"],
        name="FY"
    )
)

assert np.allclose(
    fiscal_revenue_check.to_numpy(),
    expected_fiscal_bounds.to_numpy(),
    rtol=0,
    atol=0.01
), "Fiscal revenue bounds do not reconcile."

fy_counts = (
    transaction_fy[
        ["Transaction_ID", "PossibleFYCount"]
    ]
    .drop_duplicates("Transaction_ID")
    ["PossibleFYCount"]
    .value_counts()
)

print("Fiscal-year revenue checks passed.")
print(
    "Transactions with one possible fiscal year: "
    f"{fy_counts.get(1, 0):,}"
)
print(
    "Transactions with two possible fiscal years: "
    f"{fy_counts.get(2, 0):,}"
)
print(
    "Fiscal-year route rows: "
    f"{len(route_fiscal_revenue_bounds):,}"
)

print("\nFiscal-year revenue bounds:")
display(fiscal_revenue_check.round(2))

print("\nRoute revenue bounds sample:")
display(route_fiscal_revenue_bounds.head(10).round(2))

In [ ]:
required_variables = [
    "cost_work",
    "transactions_audit",
    "date_candidates",
    "route_fiscal_revenue_bounds"
]

for name in required_variables:
    if name in globals():
        print(f"{name}: available")
    else:
        print(f"{name}: missing")

In [ ]:
from google.colab import files

integration_upload = files.upload()

print("\nUploaded files:")
for filename in integration_upload:
    print(filename)

print(f"\nTotal files uploaded: {len(integration_upload)}")

In [ ]:
import io
import pandas as pd
import numpy as np

pd.options.display.float_format = "{:,.2f}".format

# Read files from the current upload.
def read_uploaded_csv(filename):
    return pd.read_csv(
        io.BytesIO(integration_upload[filename]),
        encoding="utf-8-sig"
    )

cost_work = read_uploaded_csv("route_monthly_costs.csv")

transactions_audit = read_uploaded_csv(
    "transactions_deduplicated_audit.csv"
)

date_candidates = read_uploaded_csv(
    "transaction_date_candidates.csv"
)

gate_allocations = read_uploaded_csv(
    "NorthByte_Route_Gate_Rent_Allocations.csv"
)

gate_unallocated = read_uploaded_csv(
    "NorthByte_UnallocatedRent.csv"
)

gate_fiscal_summary = read_uploaded_csv(
    "NorthByte_Gate_Rent_FY_Summary.csv"
)

gate_reconciliation = read_uploaded_csv(
    "NorthByte_Gate_Rent_Reconciliation.csv"
)

# Check required fields.
required_columns = {
    "cost_work": {
        "Month", "Route_ID", "OperatingCost", "FlightCount"
    },
    "transactions_audit": {
        "Transaction_ID", "Route_ID", "Amount"
    },
    "date_candidates": {
        "Transaction_ID", "candidate_date",
        "candidate_fiscal_year"
    },
    "gate_allocations": {
        "Month", "Route_ID", "Scenario", "AllocatedGateRent"
    },
    "gate_unallocated": {
        "Month", "DepartureAirport", "Scenario", "UnallocatedRent"
    },
    "gate_fiscal_summary": {
        "FY", "Scenario", "TotalRent",
        "AllocatedGateRent", "UnallocatedRent"
    },
    "gate_reconciliation": {
        "Month", "DepartureAirport", "Scenario",
        "TotalRent", "AllocatedGateRent", "UnallocatedRent"
    }
}

for table_name, columns in required_columns.items():
    missing_columns = columns - set(globals()[table_name].columns)
    assert not missing_columns, (
        f"{table_name}: missing columns {missing_columns}"
    )

# Standardize directional route IDs.
def standardize_routes(series):
    codes = (
        series.astype("string")
        .str.upper()
        .str.replace(r"[^A-Z]", "", regex=True)
    )
    assert codes.notna().all(), "Missing route IDs."
    assert codes.str.fullmatch(r"[A-Z]{6}").all(), (
        "Invalid route IDs."
    )
    return codes.str[:3] + "-" + codes.str[3:]

for table in [cost_work, transactions_audit, gate_allocations]:
    table["Route_ID"] = standardize_routes(table["Route_ID"])

# Standardize month values.
for table in [
    cost_work,
    gate_allocations,
    gate_unallocated,
    gate_reconciliation
]:
    table["Month"] = (
        pd.to_datetime(table["Month"], errors="raise")
        .dt.to_period("M")
        .astype(str)
    )

# Validate numeric fields.
numeric_fields = {
    "cost_work": ["OperatingCost", "FlightCount"],
    "transactions_audit": ["Amount"],
    "gate_allocations": ["AllocatedGateRent"],
    "gate_unallocated": ["UnallocatedRent"],
    "gate_fiscal_summary": [
        "TotalRent", "AllocatedGateRent", "UnallocatedRent"
    ],
    "gate_reconciliation": [
        "TotalRent", "AllocatedGateRent", "UnallocatedRent"
    ]
}

for table_name, columns in numeric_fields.items():
    table = globals()[table_name]
    for column in columns:
        table[column] = pd.to_numeric(
            table[column], errors="raise"
        )
        assert np.isfinite(table[column]).all(), (
            f"{table_name}: invalid values in {column}"
        )

# Validate unique keys and source totals.
assert not cost_work.duplicated(["Month", "Route_ID"]).any()
assert transactions_audit["Transaction_ID"].notna().all()
assert transactions_audit["Transaction_ID"].is_unique

assert not gate_allocations.duplicated(
    ["Month", "Route_ID", "Scenario"]
).any()

assert not gate_fiscal_summary.duplicated(
    ["FY", "Scenario"]
).any()

assert not gate_reconciliation.duplicated(
    ["Month", "DepartureAirport", "Scenario"]
).any()

assert len(cost_work) == 1260
assert len(transactions_audit) == 558129
assert len(date_candidates) == 564189
assert len(gate_allocations) == 3780

assert np.isclose(
    cost_work["OperatingCost"].sum(),
    431362040.24,
    rtol=0,
    atol=0.01
), "Operating cost total does not match."

assert np.isclose(
    cost_work["FlightCount"].sum(),
    28801,
    rtol=0,
    atol=0
), "Flight count does not match."

assert np.isclose(
    transactions_audit["Amount"].sum(),
    370629024.85,
    rtol=0,
    atol=0.01
), "Net revenue total does not match."

assert np.allclose(
    gate_reconciliation["AllocatedGateRent"]
    + gate_reconciliation["UnallocatedRent"],
    gate_reconciliation["TotalRent"],
    rtol=0,
    atol=0.01
), "Gate rent reconciliation failed."

# Confirm that each rent scenario covers the cost route-months.
expected_keys = set(
    cost_work[["Month", "Route_ID"]]
    .itertuples(index=False, name=None)
)

assert set(gate_allocations["Scenario"]) == {
    "Low", "Base", "High"
}

for scenario, table in gate_allocations.groupby("Scenario"):
    actual_keys = set(
        table[["Month", "Route_ID"]]
        .itertuples(index=False, name=None)
    )
    assert actual_keys == expected_keys, (
        f"Route-month coverage mismatch: {scenario}"
    )

print("Data restoration checks passed.")
print(f"Cost route-months: {len(cost_work):,}")
print(f"Total flights: {cost_work['FlightCount'].sum():,.0f}")
print(
    "Total operating cost: "
    f"CAD {cost_work['OperatingCost'].sum():,.2f}"
)
print(f"Unique transactions: {len(transactions_audit):,}")
print(
    "Total net transaction revenue: "
    f"CAD {transactions_audit['Amount'].sum():,.2f}"
)
print(f"Gate allocation rows: {len(gate_allocations):,}")

print("\nFiscal-year gate rent summary:")
display(
    gate_fiscal_summary[
        [
            "FY", "Scenario", "TotalRent",
            "AllocatedGateRent", "UnallocatedRent"
        ]
    ].sort_values(["FY", "Scenario"]).reset_index(drop=True)
)

In [ ]:
# Validate date candidates before rebuilding revenue bounds.
assert not date_candidates.duplicated(
    ["Transaction_ID", "candidate_date"]
).any()

assert set(date_candidates["Transaction_ID"]) == set(
    transactions_audit["Transaction_ID"]
)

# Keep each transaction once per possible fiscal year.
transaction_fy = (
    date_candidates[
        ["Transaction_ID", "candidate_fiscal_year"]
    ]
    .drop_duplicates()
    .rename(columns={"candidate_fiscal_year": "FY"})
    .copy()
)

assert transaction_fy["FY"].notna().all()
assert set(transaction_fy["FY"]) == {
    "FY2024-25", "FY2025-26"
}

transaction_fy["PossibleFYCount"] = (
    transaction_fy.groupby("Transaction_ID")["FY"]
    .transform("size")
)

assert transaction_fy["PossibleFYCount"].isin([1, 2]).all()

transaction_fy = transaction_fy.merge(
    transactions_audit[["Transaction_ID", "Route_ID", "Amount"]],
    on="Transaction_ID",
    how="left",
    validate="many_to_one"
)

fy_is_certain = transaction_fy["PossibleFYCount"].eq(1)

transaction_fy["NetRevenueLower"] = np.where(
    fy_is_certain,
    transaction_fy["Amount"],
    np.minimum(transaction_fy["Amount"], 0.0)
)

transaction_fy["NetRevenueUpper"] = np.where(
    fy_is_certain,
    transaction_fy["Amount"],
    np.maximum(transaction_fy["Amount"], 0.0)
)

route_fiscal_revenue_bounds = (
    transaction_fy.groupby(["FY", "Route_ID"], as_index=False)[
        ["NetRevenueLower", "NetRevenueUpper"]
    ].sum()
)

# Reconcile FY revenue bounds.
fiscal_revenue_bounds = (
    route_fiscal_revenue_bounds.groupby("FY")[
        ["NetRevenueLower", "NetRevenueUpper"]
    ].sum().sort_index()
)

assert np.allclose(
    fiscal_revenue_bounds.to_numpy(),
    [
        [179939246.53, 181375714.90],
        [189253309.95, 190689778.32]
    ],
    rtol=0,
    atol=0.01
), "Fiscal revenue bounds do not reconcile."

# Assign fiscal years from monthly dates.
def fiscal_year_from_month(month_series):
    dates = pd.to_datetime(
        month_series, format="%Y-%m", errors="raise"
    )
    start_year = dates.dt.year - (dates.dt.month < 9).astype(int)
    return (
        "FY" + start_year.astype(str)
        + "-" + ((start_year + 1) % 100).astype(str).str.zfill(2)
    )

cost_fiscal_work = cost_work.copy()
cost_fiscal_work["FY"] = fiscal_year_from_month(
    cost_fiscal_work["Month"]
)

gate_fiscal_work = gate_allocations.copy()
gate_fiscal_work["FY"] = fiscal_year_from_month(
    gate_fiscal_work["Month"]
)

# Summarize operating costs and flights by FY and route.
route_fiscal_costs = (
    cost_fiscal_work.groupby(["FY", "Route_ID"], as_index=False)[
        ["OperatingCost", "FlightCount"]
    ].sum()
)

# Summarize allocated rent by FY, route, and scenario.
route_fiscal_gates = (
    gate_fiscal_work.groupby(
        ["FY", "Route_ID", "Scenario"],
        as_index=False
    )["AllocatedGateRent"].sum()
)

# Merge revenue and operating costs.
route_fiscal_core = route_fiscal_revenue_bounds.merge(
    route_fiscal_costs,
    on=["FY", "Route_ID"],
    how="outer",
    validate="one_to_one",
    indicator=True
)

assert route_fiscal_core["_merge"].eq("both").all(), (
    "FY route revenue and cost coverage do not match."
)

route_fiscal_core = route_fiscal_core.drop(columns="_merge")

# Add the three alternative rent scenarios.
route_fiscal_results = route_fiscal_gates.merge(
    route_fiscal_core,
    on=["FY", "Route_ID"],
    how="outer",
    validate="many_to_one",
    indicator=True
)

assert route_fiscal_results["_merge"].eq("both").all(), (
    "FY route rent coverage does not match."
)

route_fiscal_results = route_fiscal_results.drop(columns="_merge")

route_fiscal_results["AfterGateResultLower"] = (
    route_fiscal_results["NetRevenueLower"]
    - route_fiscal_results["OperatingCost"]
    - route_fiscal_results["AllocatedGateRent"]
)

route_fiscal_results["AfterGateResultUpper"] = (
    route_fiscal_results["NetRevenueUpper"]
    - route_fiscal_results["OperatingCost"]
    - route_fiscal_results["AllocatedGateRent"]
)

# Build company results using TOTAL rent, including unallocated rent.
fiscal_costs = (
    route_fiscal_costs.groupby("FY", as_index=False)[
        ["OperatingCost", "FlightCount"]
    ].sum()
)

company_core = fiscal_revenue_bounds.reset_index().merge(
    fiscal_costs,
    on="FY",
    validate="one_to_one"
)

company_fiscal_results = gate_fiscal_summary[
    ["FY", "Scenario", "TotalRent", "AllocatedGateRent", "UnallocatedRent"]
].merge(
    company_core,
    on="FY",
    validate="many_to_one"
)

company_fiscal_results["AfterGateResultLower"] = (
    company_fiscal_results["NetRevenueLower"]
    - company_fiscal_results["OperatingCost"]
    - company_fiscal_results["TotalRent"]
)

company_fiscal_results["AfterGateResultUpper"] = (
    company_fiscal_results["NetRevenueUpper"]
    - company_fiscal_results["OperatingCost"]
    - company_fiscal_results["TotalRent"]
)

# Check that route results reconcile to company results.
route_totals = (
    route_fiscal_results.groupby(["FY", "Scenario"], as_index=False)[
        [
            "AllocatedGateRent",
            "AfterGateResultLower",
            "AfterGateResultUpper"
        ]
    ].sum()
)

reconciliation_check = company_fiscal_results.merge(
    route_totals,
    on=["FY", "Scenario"],
    validate="one_to_one",
    suffixes=("_Company", "_Routes")
)

assert np.allclose(
    reconciliation_check["AllocatedGateRent_Company"],
    reconciliation_check["AllocatedGateRent_Routes"],
    rtol=0,
    atol=0.01
)

for bound in ["Lower", "Upper"]:
    assert np.allclose(
        reconciliation_check[f"AfterGateResult{bound}_Routes"]
        - reconciliation_check["UnallocatedRent"],
        reconciliation_check[f"AfterGateResult{bound}_Company"],
        rtol=0,
        atol=0.01
    ), f"Company reconciliation failed: {bound}"

print("Fiscal integration checks passed.")

print("\nCompany fiscal-year results:")
display(
    company_fiscal_results[
        [
            "FY", "Scenario",
            "NetRevenueLower", "NetRevenueUpper",
            "OperatingCost", "TotalRent",
            "AfterGateResultLower", "AfterGateResultUpper"
        ]
    ].sort_values(["FY", "Scenario"]).reset_index(drop=True)
)

print("\nFY2025-26: lowest upper results under Base rent:")
display(
    route_fiscal_results.loc[
        route_fiscal_results["FY"].eq("FY2025-26")
        & route_fiscal_results["Scenario"].eq("Base"),
        [
            "Route_ID", "FlightCount",
            "NetRevenueLower", "NetRevenueUpper",
            "OperatingCost", "AllocatedGateRent",
            "AfterGateResultLower", "AfterGateResultUpper"
        ]
    ].sort_values("AfterGateResultUpper").head(10).reset_index(drop=True)
)

In [ ]:
# Combine both directions into an airport pair.
pair_work = route_fiscal_results.copy()

pair_work["AirportPair"] = pair_work["Route_ID"].apply(
    lambda route: "-".join(sorted(route.split("-")))
)

pair_fiscal_results = (
    pair_work.groupby(
        ["FY", "Scenario", "AirportPair"],
        as_index=False
    )[
        [
            "FlightCount",
            "NetRevenueLower",
            "NetRevenueUpper",
            "OperatingCost",
            "AllocatedGateRent",
            "AfterGateResultLower",
            "AfterGateResultUpper"
        ]
    ].sum()
)

# Check that combining directions preserves all financial totals.
financial_columns = [
    "NetRevenueLower",
    "NetRevenueUpper",
    "OperatingCost",
    "AllocatedGateRent",
    "AfterGateResultLower",
    "AfterGateResultUpper"
]

direction_totals = (
    route_fiscal_results.groupby(["FY", "Scenario"])[
        financial_columns
    ].sum().sort_index()
)

pair_totals = (
    pair_fiscal_results.groupby(["FY", "Scenario"])[
        financial_columns
    ].sum().sort_index()
)

assert np.allclose(
    direction_totals.to_numpy(),
    pair_totals.to_numpy(),
    rtol=0,
    atol=0.01
), "Airport pair totals do not reconcile."

# Use all rent scenarios to classify each pair.
pair_sensitivity = (
    pair_fiscal_results.groupby(
        ["FY", "AirportPair"],
        as_index=False
    ).agg(
        WorstCaseResult=("AfterGateResultLower", "min"),
        BestCaseResult=("AfterGateResultUpper", "max"),
        ScenarioCount=("Scenario", "nunique")
    )
)

assert pair_sensitivity["ScenarioCount"].eq(3).all()

pair_sensitivity["ResultStatus"] = np.select(
    [
        pair_sensitivity["BestCaseResult"] < -0.01,
        pair_sensitivity["WorstCaseResult"] > 0.01
    ],
    [
        "Loss across all scenarios",
        "Profit across all scenarios"
    ],
    default="Break-even or scenario-sensitive"
)

# Prepare the current FY ranking under Base rent.
current_pair_results = (
    pair_fiscal_results.loc[
        pair_fiscal_results["FY"].eq("FY2025-26")
        & pair_fiscal_results["Scenario"].eq("Base")
    ]
    .merge(
        pair_sensitivity[
            [
                "FY", "AirportPair",
                "WorstCaseResult", "BestCaseResult", "ResultStatus"
            ]
        ],
        on=["FY", "AirportPair"],
        validate="one_to_one"
    )
    .sort_values("AfterGateResultUpper")
    .reset_index(drop=True)
)

assert len(current_pair_results) == 27

print("Airport pair checks passed.")
print(f"Current fiscal-year airport pairs: {len(current_pair_results)}")

print("\nCurrent FY status across all scenarios:")
display(
    current_pair_results["ResultStatus"]
    .value_counts()
    .rename_axis("ResultStatus")
    .to_frame("AirportPairCount")
)

print("\nFY2025-26: lowest upper results under Base rent:")
display(
    current_pair_results[
        [
            "AirportPair",
            "FlightCount",
            "NetRevenueLower",
            "NetRevenueUpper",
            "OperatingCost",
            "AllocatedGateRent",
            "AfterGateResultLower",
            "AfterGateResultUpper",
            "ResultStatus"
        ]
    ].head(10)
)

In [ ]:
# Select the four pairs with the lowest upper results under Base rent.
priority_pairs = (
    current_pair_results.head(4)["AirportPair"].tolist()
)

priority_pair_comparison = (
    pair_fiscal_results.loc[
        pair_fiscal_results["Scenario"].eq("Base")
        & pair_fiscal_results["AirportPair"].isin(priority_pairs)
    ]
    .copy()
)

assert priority_pair_comparison["FlightCount"].gt(0).all()
assert len(priority_pair_comparison) == 8

# Calculate annual averages per recorded flight.
priority_pair_comparison["RevenuePerFlightLower"] = (
    priority_pair_comparison["NetRevenueLower"]
    / priority_pair_comparison["FlightCount"]
)

priority_pair_comparison["RevenuePerFlightUpper"] = (
    priority_pair_comparison["NetRevenueUpper"]
    / priority_pair_comparison["FlightCount"]
)

priority_pair_comparison["OperatingCostPerFlight"] = (
    priority_pair_comparison["OperatingCost"]
    / priority_pair_comparison["FlightCount"]
)

priority_pair_comparison["GateRentPerFlight"] = (
    priority_pair_comparison["AllocatedGateRent"]
    / priority_pair_comparison["FlightCount"]
)

priority_pair_comparison["ResultPerFlightLower"] = (
    priority_pair_comparison["AfterGateResultLower"]
    / priority_pair_comparison["FlightCount"]
)

priority_pair_comparison["ResultPerFlightUpper"] = (
    priority_pair_comparison["AfterGateResultUpper"]
    / priority_pair_comparison["FlightCount"]
)

priority_pair_comparison = (
    priority_pair_comparison.sort_values(["AirportPair", "FY"])
    .reset_index(drop=True)
)

print("Priority pair comparison under Base rent:")
display(
    priority_pair_comparison[
        [
            "AirportPair",
            "FY",
            "FlightCount",
            "NetRevenueLower",
            "NetRevenueUpper",
            "OperatingCost",
            "AllocatedGateRent",
            "AfterGateResultLower",
            "AfterGateResultUpper"
        ]
    ]
)

print("\nAnnual financial averages per recorded flight:")
display(
    priority_pair_comparison[
        [
            "AirportPair",
            "FY",
            "FlightCount",
            "RevenuePerFlightLower",
            "RevenuePerFlightUpper",
            "OperatingCostPerFlight",
            "GateRentPerFlight",
            "ResultPerFlightLower",
            "ResultPerFlightUpper"
        ]
    ].round(2)
)

In [ ]:
# Compare the two fiscal years for the four priority airport pairs.
comparison_columns = [
    "AirportPair",
    "FlightCount",
    "NetRevenueLower",
    "NetRevenueUpper",
    "OperatingCost",
    "AllocatedGateRent",
    "AfterGateResultLower",
    "AfterGateResultUpper",
    "OperatingCostPerFlight"
]

previous = priority_pair_comparison.loc[
    priority_pair_comparison["FY"].eq("FY2024-25"),
    comparison_columns
]

current = priority_pair_comparison.loc[
    priority_pair_comparison["FY"].eq("FY2025-26"),
    comparison_columns
]

priority_pair_changes = previous.merge(
    current,
    on="AirportPair",
    validate="one_to_one",
    suffixes=("_Previous", "_Current")
)

priority_pair_changes["FlightIncrease"] = (
    priority_pair_changes["FlightCount_Current"]
    - priority_pair_changes["FlightCount_Previous"]
)

priority_pair_changes["FlightGrowthPct"] = (
    priority_pair_changes["FlightIncrease"]
    / priority_pair_changes["FlightCount_Previous"]
    * 100
)

# Conservative change bounds: current lower minus previous upper,
# and current upper minus previous lower.
priority_pair_changes["RevenueChangeLower"] = (
    priority_pair_changes["NetRevenueLower_Current"]
    - priority_pair_changes["NetRevenueUpper_Previous"]
)

priority_pair_changes["RevenueChangeUpper"] = (
    priority_pair_changes["NetRevenueUpper_Current"]
    - priority_pair_changes["NetRevenueLower_Previous"]
)

priority_pair_changes["OperatingCostIncrease"] = (
    priority_pair_changes["OperatingCost_Current"]
    - priority_pair_changes["OperatingCost_Previous"]
)

priority_pair_changes["CostPerFlightGrowthPct"] = (
    priority_pair_changes["OperatingCostPerFlight_Current"]
    / priority_pair_changes["OperatingCostPerFlight_Previous"]
    - 1
) * 100

priority_pair_changes["AllocatedRentChange"] = (
    priority_pair_changes["AllocatedGateRent_Current"]
    - priority_pair_changes["AllocatedGateRent_Previous"]
)

# Positive values indicate a larger financial deficit.
priority_pair_changes["DeficitIncreaseLower"] = (
    priority_pair_changes["AfterGateResultLower_Previous"]
    - priority_pair_changes["AfterGateResultUpper_Current"]
)

priority_pair_changes["DeficitIncreaseUpper"] = (
    priority_pair_changes["AfterGateResultUpper_Previous"]
    - priority_pair_changes["AfterGateResultLower_Current"]
)

priority_pair_changes = priority_pair_changes.sort_values(
    "DeficitIncreaseLower",
    ascending=False
).reset_index(drop=True)

print("Priority airport pair changes under Base rent:")
display(
    priority_pair_changes[
        [
            "AirportPair",
            "FlightIncrease",
            "FlightGrowthPct",
            "RevenueChangeLower",
            "RevenueChangeUpper",
            "OperatingCostIncrease",
            "CostPerFlightGrowthPct",
            "AllocatedRentChange",
            "DeficitIncreaseLower",
            "DeficitIncreaseUpper"
        ]
    ].round(2)
)

Key findings

FY2025–26 shows a financial deficit of CAD 57.71–59.15 million
under the Base gate-rent scenario. The deficit remains negative
under all tested revenue-date allocations and gate-rent scenarios.

Of the 27 airport pairs, 15 have negative results across all tested
scenarios, while 12 have positive results across all tested scenarios.

Four airport pairs warrant priority review:
Montreal–Vancouver, Halifax–Toronto, Montreal–Toronto,
and Ottawa–Toronto.

Together, these services added 1,189 flights. Operating costs
increased by CAD 26.84 million, while net transaction revenue
increased by only CAD 0.80–1.46 million.

Recommended actions

1. Pause further frequency expansion on the four priority services
   pending a flight-level commercial and network review.

2. Investigate increases in fuel, crew, maintenance, and food costs
   per flight, and identify controllable cost drivers.

3. Review the operational need and contractual flexibility of
   unassigned gates, and pursue evidence-based lease negotiations.

Interpretation limits

Revenue is grouped by transaction date, while operating costs are
grouped by flight date. These results are fiscal-period financial
comparisons, not matched flight-level profitability.

Allocated gate rent is not necessarily avoidable. Historical deficits
and cost increases must not be presented as guaranteed savings
from cancelling flights or routes.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from pathlib import Path

chart_data = priority_pair_changes.sort_values(
    "OperatingCostIncrease",
    ascending=False
).copy()

pair_labels = {
    "YUL-YVR": "Montreal–Vancouver",
    "YHZ-YYZ": "Halifax–Toronto",
    "YUL-YYZ": "Montreal–Toronto",
    "YOW-YYZ": "Ottawa–Toronto"
}

labels = chart_data["AirportPair"].map(pair_labels).tolist()
cost = chart_data["OperatingCostIncrease"].to_numpy() / 1_000_000
lower = chart_data["RevenueChangeLower"].to_numpy() / 1_000_000
upper = chart_data["RevenueChangeUpper"].to_numpy() / 1_000_000

y = np.arange(len(chart_data))

fig, ax = plt.subplots(figsize=(12, 6.5))
fig.subplots_adjust(left=0.23, right=0.96, top=0.80, bottom=0.25)

# Show operating cost increases as bars.
ax.barh(
    y - 0.17,
    cost,
    height=0.28,
    color="#D95F45"
)

# Show revenue increases as ranges, without midpoint estimates.
revenue_y = y + 0.17

ax.hlines(
    revenue_y,
    lower,
    upper,
    color="#287D8E",
    linewidth=4
)

ax.scatter(
    lower, revenue_y,
    color="#287D8E", marker="|", s=120, linewidths=2
)

ax.scatter(
    upper, revenue_y,
    color="#287D8E", marker="|", s=120, linewidths=2
)

for i in range(len(chart_data)):
    ax.text(
        cost[i] + 0.15,
        y[i] - 0.17,
        f"{cost[i]:.2f}m",
        va="center",
        fontsize=11
    )

    ax.text(
        upper[i] + 0.18,
        revenue_y[i],
        f"{lower[i]:.2f}–{upper[i]:.2f}m",
        va="center",
        fontsize=11,
        color="#205F6B"
    )

ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=12)
ax.invert_yaxis()
ax.set_xlim(0, 14)
ax.set_xlabel("Annual increase (CAD millions)", fontsize=11)
ax.tick_params(axis="x", labelsize=10)

ax.grid(axis="x", alpha=0.18)
ax.set_axisbelow(True)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

ax.tick_params(axis="y", length=0)

fig.text(
    0.23, 0.94,
    "Cost growth far exceeded revenue growth",
    fontsize=18, fontweight="bold"
)

fig.text(
    0.23, 0.89,
    "Four priority airport pairs | FY2025–26 versus FY2024–25",
    fontsize=12, color="#555555"
)

legend_items = [
    Patch(
        facecolor="#D95F45",
        label="Operating cost increase"
    ),
    Line2D(
        [0], [0],
        color="#287D8E", linewidth=4,
        label="Net transaction revenue increase: lower–upper bounds"
    )
]

fig.legend(
    handles=legend_items,
    loc="lower left",
    bbox_to_anchor=(0.23, 0.11),
    frameon=False,
    fontsize=10
)

fig.text(
    0.23, 0.045,
    "Bounds reflect date-allocation uncertainty, not confidence intervals.\n"
    "Revenue uses transaction dates; costs use flight dates. "
    "Changes do not measure added-flight profitability.",
    fontsize=9,
    color="#555555"
)

output_directory = Path("/content/integration_outputs")
output_directory.mkdir(parents=True, exist_ok=True)

chart_path = (
    output_directory / "01_priority_cost_vs_revenue_growth.png"
)

fig.savefig(
    chart_path,
    dpi=220,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
print(f"Chart saved: {chart_path}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Prepare company deficit ranges for each rent scenario.
company_chart = company_fiscal_results.copy()

assert company_chart["AfterGateResultUpper"].lt(0).all()

company_chart["DeficitLowerMillions"] = (
    -company_chart["AfterGateResultUpper"] / 1_000_000
)

company_chart["DeficitUpperMillions"] = (
    -company_chart["AfterGateResultLower"] / 1_000_000
)

fiscal_years = ["FY2024-25", "FY2025-26"]
scenarios = ["Low", "Base", "High"]

colors = {
    "Low": "#287D8E",
    "Base": "#D95F45",
    "High": "#74549A"
}

offsets = {
    "Low": -0.20,
    "Base": 0.00,
    "High": 0.20
}

fig, ax = plt.subplots(figsize=(12, 6))
fig.subplots_adjust(
    left=0.16, right=0.96, top=0.78, bottom=0.32
)

for scenario in scenarios:
    rows = (
        company_chart.loc[
            company_chart["Scenario"].eq(scenario)
        ]
        .set_index("FY")
        .loc[fiscal_years]
    )

    lower = rows["DeficitLowerMillions"].to_numpy()
    upper = rows["DeficitUpperMillions"].to_numpy()
    y = np.arange(len(fiscal_years)) + offsets[scenario]

    ax.hlines(
        y, lower, upper,
        color=colors[scenario],
        linewidth=5,
        label=f"{scenario} rent"
    )

    ax.scatter(
        lower, y,
        marker="|", s=150,
        color=colors[scenario],
        linewidths=2
    )

    ax.scatter(
        upper, y,
        marker="|", s=150,
        color=colors[scenario],
        linewidths=2
    )

    for i in range(len(fiscal_years)):
        ax.text(
            upper[i] + 1.0,
            y[i],
            f"{lower[i]:.2f}–{upper[i]:.2f}m",
            va="center",
            fontsize=11,
            color=colors[scenario]
        )

ax.set_yticks(np.arange(len(fiscal_years)))
ax.set_yticklabels(fiscal_years, fontsize=13)
ax.set_ylim(1.45, -0.45)
ax.set_xlim(0, 72)

ax.set_xlabel(
    "Financial deficit after gate rent (CAD millions)",
    fontsize=12
)

ax.grid(axis="x", alpha=0.18)
ax.set_axisbelow(True)
ax.tick_params(axis="y", length=0)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

fig.text(
    0.16, 0.94,
    "The financial deficit widened in every tested scenario",
    fontsize=17,
    fontweight="bold"
)

fig.text(
    0.16, 0.88,
    "Revenue date-allocation bounds with Low, Base and High gate-rent estimates",
    fontsize=11,
    color="#555555"
)

ax.legend(
    loc="upper left",
    bbox_to_anchor=(0, -0.12),
    ncol=3,
    frameon=False,
    fontsize=11
)

fig.text(
    0.16, 0.045,
    "Positive values show deficit magnitude; larger values indicate worse results.\n"
    "Includes all gate rent, including unallocated rent. "
    "Ranges are scenario bounds, not confidence intervals.\n"
    "Revenue uses transaction dates; operating costs use flight dates.",
    fontsize=9,
    color="#555555"
)

output_directory = Path("/content/integration_outputs")
output_directory.mkdir(parents=True, exist_ok=True)

company_chart_path = (
    output_directory / "02_company_fiscal_deficit_ranges.png"
)

fig.savefig(
    company_chart_path,
    dpi=220,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()
print(f"Chart saved: {company_chart_path}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Select the four priority airport pairs under Base rent.
pair_chart = (
    current_pair_results.head(4)
    .copy()
    .sort_values("AfterGateResultUpper")
    .reset_index(drop=True)
)

assert pair_chart["AfterGateResultUpper"].lt(0).all()

pair_labels = {
    "YUL-YVR": "Montreal–Vancouver",
    "YHZ-YYZ": "Halifax–Toronto",
    "YUL-YYZ": "Montreal–Toronto",
    "YOW-YYZ": "Ottawa–Toronto"
}

labels = pair_chart["AirportPair"].map(pair_labels).tolist()

# Convert negative results into positive deficit magnitudes.
deficit_lower = (
    -pair_chart["AfterGateResultUpper"].to_numpy() / 1_000_000
)

deficit_upper = (
    -pair_chart["AfterGateResultLower"].to_numpy() / 1_000_000
)

y = np.arange(len(pair_chart))

bar_color = "#E85D3F"
range_color = "#16324F"

fig, ax = plt.subplots(figsize=(12, 6))

fig.subplots_adjust(
    left=0.23,
    right=0.96,
    top=0.80,
    bottom=0.23
)

# Orange bars show the minimum deficit.
ax.barh(
    y,
    deficit_lower,
    height=0.40,
    color=bar_color
)

# Navy segments show the full date-allocation range.
ax.hlines(
    y,
    deficit_lower,
    deficit_upper,
    color=range_color,
    linewidth=6
)

for endpoint in [deficit_lower, deficit_upper]:
    ax.scatter(
        endpoint,
        y,
        marker="|",
        s=160,
        linewidths=2,
        color=range_color,
        zorder=3
    )

# Add financial range labels.
for i in range(len(pair_chart)):
    ax.text(
        deficit_upper[i] + 0.35,
        y[i],
        f"{deficit_lower[i]:.2f}–{deficit_upper[i]:.2f}m",
        va="center",
        fontsize=12,
        fontweight="bold",
        color=range_color
    )

ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=12)
ax.invert_yaxis()
ax.set_xlim(0, 25)

ax.set_xlabel(
    "Financial deficit after allocated gate rent (CAD millions)",
    fontsize=11
)

ax.grid(axis="x", alpha=0.18)
ax.set_axisbelow(True)
ax.tick_params(axis="y", length=0)
ax.tick_params(axis="x", labelsize=10)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

fig.text(
    0.23,
    0.94,
    "Montreal–Vancouver has the largest airport-pair deficit",
    fontsize=17,
    fontweight="bold",
    color=range_color
)

fig.text(
    0.23,
    0.88,
    "Four largest deficits | FY2025–26 | Base gate-rent scenario",
    fontsize=12,
    color="#555555"
)

fig.text(
    0.23,
    0.07,
    "Orange bars show minimum deficits; navy segments show date-allocation ranges.\n"
    "Revenue uses transaction dates; costs use flight dates. "
    "Deficits are not guaranteed savings from route closure.",
    fontsize=9,
    color="#555555"
)

# Save the updated chart.
output_directory = Path("/content/integration_outputs")
output_directory.mkdir(parents=True, exist_ok=True)

pair_chart_path = (
    output_directory / "03_priority_airport_pair_deficits.png"
)

fig.savefig(
    pair_chart_path,
    dpi=220,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print(f"Chart saved: {pair_chart_path}")

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

output_directory = Path("/content/integration_outputs")
output_directory.mkdir(parents=True, exist_ok=True)

# Export the integrated result tables.
result_tables = {
    "company_fiscal_results.csv": company_fiscal_results,
    "route_fiscal_results.csv": route_fiscal_results,
    "pair_fiscal_results.csv": pair_fiscal_results,
    "pair_sensitivity.csv": pair_sensitivity,
    "current_pair_results.csv": current_pair_results,
    "priority_pair_comparison.csv": priority_pair_comparison,
    "priority_pair_changes.csv": priority_pair_changes,
    "route_fiscal_revenue_bounds.csv": route_fiscal_revenue_bounds
}

for filename, table in result_tables.items():
    table.to_csv(
        output_directory / filename,
        index=False,
        encoding="utf-8-sig"
    )

# Document the main interpretation rules.
readme = """First Byte Datathon - Integrated Analysis

Period: September 2024 to August 2026.
Currency: CAD.
Fiscal years run from September to August.

Contents:
- Eight integrated result tables.
- Three presentation charts.

Interpretation:
- Revenue bounds reflect possible transaction-date assignments.
- These bounds are not statistical confidence intervals.
- Low, Base and High rent scenarios are alternatives; do not add them.
- Company results deduct total rent, including unallocated rent.
- Route results deduct allocated rent only.
- Revenue uses transaction dates; operating costs use flight dates.
- Results are fiscal-period financial comparisons, not matched
  flight-level profitability.
- Allocated rent is not necessarily avoidable.
- Historical deficits are not guaranteed savings from route closure.
"""

(output_directory / "README.txt").write_text(
    readme,
    encoding="utf-8"
)

# Confirm that all three chart files exist.
chart_filenames = [
    "01_priority_cost_vs_revenue_growth.png",
    "02_company_fiscal_deficit_ranges.png",
    "03_priority_airport_pair_deficits.png"
]

for filename in chart_filenames:
    assert (output_directory / filename).is_file(), (
        f"Missing chart: {filename}"
    )

# Package only the intended deliverables.
package_filenames = (
    list(result_tables)
    + chart_filenames
    + ["README.txt"]
)

zip_path = Path("/content/FirstByte_Integrated_Results.zip")

with ZipFile(zip_path, "w", ZIP_DEFLATED) as archive:
    for filename in package_filenames:
        archive.write(
            output_directory / filename,
            arcname=filename
        )

print(f"Package created: {zip_path.name}")
print(f"Files included: {len(package_filenames)}")

files.download(str(zip_path))

In [ ]:
# Prepare the current-year baseline for the four priority airport pairs.
action_baseline = current_pair_results.loc[
    current_pair_results["AirportPair"].isin(
        priority_pair_changes["AirportPair"]
    ),
    [
        "AirportPair",
        "FlightCount",
        "NetRevenueLower",
        "NetRevenueUpper",
        "OperatingCost",
        "AllocatedGateRent",
        "AfterGateResultLower",
        "AfterGateResultUpper"
    ]
].copy()

assert len(action_baseline) == 4
assert action_baseline["FlightCount"].gt(0).all()

action_baseline["OperatingCostPerFlight"] = (
    action_baseline["OperatingCost"]
    / action_baseline["FlightCount"]
)

action_baseline["RevenuePerFlightLower"] = (
    action_baseline["NetRevenueLower"]
    / action_baseline["FlightCount"]
)

action_baseline["RevenuePerFlightUpper"] = (
    action_baseline["NetRevenueUpper"]
    / action_baseline["FlightCount"]
)

# Screening threshold under an illustrative assumption:
# revenue lost per removed flight equals current annual average revenue.
# Gate rent remains unchanged.
action_baseline["AvoidableCostThresholdLowerPct"] = (
    action_baseline["RevenuePerFlightLower"]
    / action_baseline["OperatingCostPerFlight"]
    * 100
)

action_baseline["AvoidableCostThresholdUpperPct"] = (
    action_baseline["RevenuePerFlightUpper"]
    / action_baseline["OperatingCostPerFlight"]
    * 100
)

action_baseline = action_baseline.sort_values(
    "AfterGateResultUpper"
).reset_index(drop=True)

print("Frequency-adjustment screening baseline:")
display(
    action_baseline[
        [
            "AirportPair",
            "FlightCount",
            "OperatingCostPerFlight",
            "RevenuePerFlightLower",
            "RevenuePerFlightUpper",
            "AvoidableCostThresholdLowerPct",
            "AvoidableCostThresholdUpperPct"
        ]
    ].round(2)
)

print(
    "\nIllustrative screening only: "
    "annual average revenue is not matched flight revenue."
)
print(
    "Gate rent is assumed unchanged. "
    "Network effects and implementation costs are not included."
)

In [ ]:
# Illustrative frequency-reduction sensitivity analysis.
reduction_target = 0.10
avoidable_cost_rates = [0.40, 0.60, 0.80]

scenario_rows = []

for _, row in action_baseline.iterrows():
    # Remove flights in pairs to preserve equal directional reductions.
    removed_flights = int(
        np.floor(row["FlightCount"] * reduction_target / 2) * 2
    )

    actual_reduction_rate = removed_flights / row["FlightCount"]

    revenue_lost_lower = (
        actual_reduction_rate * row["NetRevenueLower"]
    )

    revenue_lost_upper = (
        actual_reduction_rate * row["NetRevenueUpper"]
    )

    for avoidable_rate in avoidable_cost_rates:
        operating_cost_saved = (
            actual_reduction_rate
            * row["OperatingCost"]
            * avoidable_rate
        )

        scenario_rows.append({
            "AirportPair": row["AirportPair"],
            "RemovedFlights": removed_flights,
            "FlightReductionPct": actual_reduction_rate * 100,
            "AvoidableCostPct": avoidable_rate * 100,
            "OperatingCostSaved": operating_cost_saved,
            "RevenueLostLower": revenue_lost_lower,
            "RevenueLostUpper": revenue_lost_upper,
            "ImprovementLower": (
                operating_cost_saved - revenue_lost_upper
            ),
            "ImprovementUpper": (
                operating_cost_saved - revenue_lost_lower
            )
        })

frequency_scenarios = pd.DataFrame(scenario_rows)

frequency_scenarios["ScenarioStatus"] = np.select(
    [
        frequency_scenarios["ImprovementLower"] > 0.01,
        frequency_scenarios["ImprovementUpper"] < -0.01
    ],
    [
        "Improves across revenue bounds",
        "Worsens across revenue bounds"
    ],
    default="Break-even or date-sensitive"
)

frequency_scenario_totals = (
    frequency_scenarios.groupby(
        "AvoidableCostPct",
        as_index=False
    )[
        [
            "RemovedFlights",
            "OperatingCostSaved",
            "RevenueLostLower",
            "RevenueLostUpper",
            "ImprovementLower",
            "ImprovementUpper"
        ]
    ].sum()
)

print("Illustrative scenarios by airport pair:")
display(
    frequency_scenarios[
        [
            "AirportPair",
            "RemovedFlights",
            "FlightReductionPct",
            "AvoidableCostPct",
            "ImprovementLower",
            "ImprovementUpper",
            "ScenarioStatus"
        ]
    ].round(2)
)

print("\nCombined impact across the four airport pairs:")
display(frequency_scenario_totals.round(2))

print(
    "\nAssumptions: removed flights have annual average revenue "
    "and operating cost; gate rent is unchanged."
)
print(
    "Even flight counts support a paired-removal assumption, "
    "but do not establish a feasible timetable."
)
print(
    "Network effects, revenue recapture, implementation costs "
    "and customer impacts are excluded."
)

In [ ]:
from pathlib import Path

output_directory = Path("/content/integration_outputs")
output_directory.mkdir(parents=True, exist_ok=True)

action_baseline.to_csv(
    output_directory / "frequency_adjustment_baseline.csv",
    index=False,
    encoding="utf-8-sig"
)

frequency_scenarios.to_csv(
    output_directory / "frequency_adjustment_scenarios.csv",
    index=False,
    encoding="utf-8-sig"
)

frequency_scenario_totals.to_csv(
    output_directory / "frequency_adjustment_totals.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Frequency-adjustment analysis saved.")
print("Three new CSV files were added to integration_outputs.")
print("The previously downloaded ZIP has not been updated.")

In [ ]:
required_cost_columns = [
    "Month",
    "Route_ID",
    "FlightCount",
    "OperatingCost",
    "FuelCost",
    "CrewCost",
    "MaintenanceCost",
    "FoodCost"
]

print("Cost column availability:")

for column in required_cost_columns:
    status = "available" if column in cost_work.columns else "missing"
    print(f"{column}: {status}")

print("\nCost data sample:")
display(cost_work.head(3))

In [ ]:
# Prepare monthly costs without changing the existing cost table.
category_work = cost_work.copy()

category_work["FY"] = fiscal_year_from_month(
    category_work["Month"]
)

category_work["AirportPair"] = category_work["Route_ID"].apply(
    lambda route: "-".join(sorted(route.split("-")))
)

cost_categories = [
    "FuelCost",
    "CrewCost",
    "MaintenanceCost",
    "FoodCost"
]

# Validate category amounts and reconcile them to operating costs.
for column in cost_categories:
    category_work[column] = pd.to_numeric(
        category_work[column],
        errors="raise"
    )
    assert np.isfinite(category_work[column]).all()

assert np.allclose(
    category_work[cost_categories].sum(axis=1),
    category_work["OperatingCost"],
    rtol=0,
    atol=0.02
), "Cost categories do not reconcile to operating costs."

# Summarize the four priority pairs by fiscal year.
priority_cost_summary = (
    category_work.loc[
        category_work["AirportPair"].isin(
            action_baseline["AirportPair"]
        )
    ]
    .groupby(["AirportPair", "FY"], as_index=False)[
        cost_categories + ["OperatingCost", "FlightCount"]
    ]
    .sum()
)

assert len(priority_cost_summary) == 8
assert priority_cost_summary["FlightCount"].gt(0).all()

previous_costs = (
    priority_cost_summary.loc[
        priority_cost_summary["FY"].eq("FY2024-25")
    ].set_index("AirportPair")
)

current_costs = (
    priority_cost_summary.loc[
        priority_cost_summary["FY"].eq("FY2025-26")
    ].set_index("AirportPair")
)

category_rows = []

for pair in action_baseline["AirportPair"]:
    previous = previous_costs.loc[pair]
    current = current_costs.loc[pair]

    total_cost_increase = (
        current["OperatingCost"] - previous["OperatingCost"]
    )

    assert total_cost_increase > 0

    for category in cost_categories:
        previous_per_flight = (
            previous[category] / previous["FlightCount"]
        )
        current_per_flight = (
            current[category] / current["FlightCount"]
        )

        increase = current[category] - previous[category]

        category_rows.append({
            "AirportPair": pair,
            "CostCategory": category,
            "PreviousCost": previous[category],
            "CurrentCost": current[category],
            "CostIncrease": increase,
            "ShareOfPairCostIncreasePct": (
                increase / total_cost_increase * 100
            ),
            "PreviousCostPerFlight": previous_per_flight,
            "CurrentCostPerFlight": current_per_flight,
            "CostPerFlightGrowthPct": (
                current_per_flight / previous_per_flight - 1
            ) * 100
        })

priority_cost_category_changes = pd.DataFrame(category_rows)

# Check category increases against the earlier pair comparison.
category_increase_check = (
    priority_cost_category_changes.groupby("AirportPair")[
        "CostIncrease"
    ].sum().sort_index()
)

expected_increase = (
    priority_pair_changes.set_index("AirportPair")[
        "OperatingCostIncrease"
    ].sort_index()
)

assert np.allclose(
    category_increase_check.to_numpy(),
    expected_increase.to_numpy(),
    rtol=0,
    atol=0.02
), "Category increases do not match the earlier analysis."

# Summarize category increases across all four pairs.
priority_category_totals = (
    priority_cost_category_changes.groupby(
        "CostCategory",
        as_index=False
    )[["PreviousCost", "CurrentCost", "CostIncrease"]]
    .sum()
    .sort_values("CostIncrease", ascending=False)
    .reset_index(drop=True)
)

priority_category_totals["ShareOfTotalIncreasePct"] = (
    priority_category_totals["CostIncrease"]
    / priority_category_totals["CostIncrease"].sum()
    * 100
)

print("Priority cost category checks passed.")

print("\nCombined cost increases across the four priority pairs:")
display(priority_category_totals.round(2))

print("\nCategory increases by airport pair:")
display(
    priority_cost_category_changes[
        [
            "AirportPair",
            "CostCategory",
            "CostIncrease",
            "ShareOfPairCostIncreasePct"
        ]
    ].round(2)
)

print("\nAverage category costs per recorded flight:")
display(
    priority_cost_category_changes[
        [
            "AirportPair",
            "CostCategory",
            "PreviousCostPerFlight",
            "CurrentCostPerFlight",
            "CostPerFlightGrowthPct"
        ]
    ].round(2)
)

# Save the new analysis.
output_directory = Path("/content/integration_outputs")
output_directory.mkdir(parents=True, exist_ok=True)

priority_cost_category_changes.to_csv(
    output_directory / "priority_cost_category_changes.csv",
    index=False,
    encoding="utf-8-sig"
)

priority_category_totals.to_csv(
    output_directory / "priority_cost_category_totals.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nTwo cost analysis CSV files saved.")

In [ ]:
# Identify pairs with positive results across all tested scenarios.
positive_pair_results = (
    current_pair_results.loc[
        current_pair_results["ResultStatus"].eq(
            "Profit across all scenarios"
        )
    ]
    .copy()
    .sort_values("AfterGateResultLower", ascending=False)
    .reset_index(drop=True)
)

assert len(positive_pair_results) == 12

print("FY2025-26 positive airport pairs under Base rent:")
display(
    positive_pair_results[
        [
            "AirportPair",
            "FlightCount",
            "NetRevenueLower",
            "NetRevenueUpper",
            "OperatingCost",
            "AllocatedGateRent",
            "AfterGateResultLower",
            "AfterGateResultUpper"
        ]
    ].round(2)
)

# Summarize positive and negative groups without netting them together.
network_group_summary = (
    current_pair_results.groupby(
        "ResultStatus",
        as_index=False
    ).agg(
        AirportPairCount=("AirportPair", "size"),
        FlightCount=("FlightCount", "sum"),
        AfterGateResultLower=("AfterGateResultLower", "sum"),
        AfterGateResultUpper=("AfterGateResultUpper", "sum")
    )
)

# Check that group results reconcile to the company Base scenario.
company_base = company_fiscal_results.loc[
    company_fiscal_results["FY"].eq("FY2025-26")
    & company_fiscal_results["Scenario"].eq("Base")
]

assert len(company_base) == 1

for bound in ["Lower", "Upper"]:
    assert np.isclose(
        network_group_summary[f"AfterGateResult{bound}"].sum()
        - company_base["UnallocatedRent"].iloc[0],
        company_base[f"AfterGateResult{bound}"].iloc[0],
        rtol=0,
        atol=0.01
    ), f"Network group reconciliation failed: {bound}"

print("\nNetwork group financial results under Base rent:")
display(network_group_summary.round(2))

positive_pair_results.to_csv(
    output_directory / "positive_airport_pair_results.csv",
    index=False,
    encoding="utf-8-sig"
)

network_group_summary.to_csv(
    output_directory / "network_group_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nNetwork analysis checks passed. Two CSV files saved.")


In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

output_directory = Path("/content/integration_outputs")
output_directory.mkdir(parents=True, exist_ok=True)

# Ensure frequency-adjustment results are saved.
additional_tables = {
    "frequency_adjustment_baseline.csv": action_baseline,
    "frequency_adjustment_scenarios.csv": frequency_scenarios,
    "frequency_adjustment_totals.csv": frequency_scenario_totals,
    "priority_cost_category_changes.csv": priority_cost_category_changes,
    "priority_cost_category_totals.csv": priority_category_totals,
    "positive_airport_pair_results.csv": positive_pair_results,
    "network_group_summary.csv": network_group_summary
}

for filename, table in additional_tables.items():
    table.to_csv(
        output_directory / filename,
        index=False,
        encoding="utf-8-sig"
    )

# Add interpretation notes for the new analysis.
additional_notes = """Additional analysis notes

Priority cost categories:
Cost increases and per-flight amounts are descriptive.
They do not establish changes in fuel prices, wages, or causal drivers.

Frequency-adjustment scenarios:
These are illustrative annual scenarios, not validated savings forecasts.
Removed flights are assumed to have annual average revenue and cost.
Avoidable operating cost rates of 40%, 60%, and 80% are assumptions.
Gate rent remains unchanged.
Network effects, revenue recapture, and implementation costs are excluded.
Paired flight removals do not establish a feasible timetable.

Network results:
Positive and negative groups describe fiscal-period financial results.
They do not establish specific connecting-passenger subsidy relationships.
Positive historical results do not prove that added flights will be profitable.
"""

(output_directory / "ADDITIONAL_ANALYSIS_NOTES.txt").write_text(
    additional_notes,
    encoding="utf-8"
)

# Package the existing and newly saved outputs.
package_files = sorted(
    path for path in output_directory.iterdir()
    if path.is_file() and path.suffix.lower() in {".csv", ".png", ".txt"}
)

zip_path = Path("/content/FirstByte_Integrated_Results_v2.zip")

with ZipFile(zip_path, "w", ZIP_DEFLATED) as archive:
    for path in package_files:
        archive.write(path, arcname=path.name)

print(f"Package created: {zip_path.name}")
print(f"Files included: {len(package_files)}")

files.download(str(zip_path))